In [1]:
%load_ext autoreload
%autoreload 2
from src.load import load_prior_lines, load_products
from src.pairs import orders_per_user, find_regulars, run_batches, compute_lift, add_names

lines = load_prior_lines()
products = load_products()
n_orders = orders_per_user(lines)
reg = find_regulars(lines, n_orders)
print(lines.shape, len(reg))

(32434489, 6) 223366


In [2]:
totals = run_batches(lines, reg, n_batches=10)

batch 1/10 done
batch 2/10 done
batch 3/10 done
batch 4/10 done
batch 5/10 done
batch 6/10 done
batch 7/10 done
batch 8/10 done
batch 9/10 done
batch 10/10 done


In [3]:
pairs = add_names(compute_lift(totals), products)
print(len(pairs))
pairs[["a_name", "b_name", "k_m", "n_m", "k_b", "n_b", "lift"]].head(20)

209725


,a_name,b_name,k_m,n_m,k_b,n_b,lift
0,Water Mineral,Mineral Water,32,443,0.0,1092,78.952596
1,Total 2% All Natural Low Fat 2% Milkfat Greek ...,Total 2% Lowfat Plain Greek Yogurt,40,1338,0.0,2483,74.260090
2,Organic Large Brown Grade AA Cage Free Eggs,Organic Large Grade AA Brown Eggs,44,4076,0.0,6556,70.782139
3,Small Hass Avocado,Organic Hass Avocado,87,2706,1.0,4097,65.876940
4,Bartlett Pears,Organic Bartlett Pear,42,2140,0.0,3315,65.080374
5,Organic Banana,Bag of Organic Bananas,40,713,0.0,1121,62.945302
6,Banana,Bag of Organic Bananas,2404,95295,84.0,207459,61.571509
7,Organic Reduced Fat 2% Milk,"Organic Milk Reduced Fat, 2% Milkfat",59,7974,1.0,15862,58.685541
8,Michigan Organic Kale,Kale Greens,31,6006,0.0,10973,56.642358
9,Organic Baby Carrots,Organic Peeled Whole Baby Carrots,96,4098,2.0,6362,49.686676


In [4]:
from src.pairs import add_lift_lower

ranked = add_lift_lower(pairs)
ranked[["a_name", "b_name", "k_m", "k_b", "lift", "lift_low"]].head(20)

,a_name,b_name,k_m,k_b,lift,lift_low
0,Banana,Bag of Organic Bananas,2404,84.0,61.571509,49.601332
1,Bag of Organic Bananas,Banana,1454,60.0,46.649167,36.112975
2,Organic Hass Avocado,Organic Avocado,712,29.0,38.522400,26.741496
3,Bag of Organic Bananas,Organic Banana,542,27.0,37.883542,25.913704
4,Organic Strawberries,Strawberries,1043,44.0,34.113610,25.321371
5,Banana,Organic Banana,260,13.0,40.430543,23.615184
6,Organic Avocado,Organic Hass Avocado,722,39.0,31.497026,22.917812
7,Organic Baby Spinach,Baby Spinach,321,17.0,29.341582,18.255617
8,Cucumber Kirby,Organic Cucumber,237,12.0,30.451046,17.435142
9,Organic Lemon,Large Lemon,139,4.0,41.387390,16.971489


In [5]:
r = ranked[ranked["a_name"] == "Banana"]
print((r["lift_low"] > 1).sum(), len(r))
r[["b_name", "k_m", "k_b", "lift", "lift_low"]].tail(5)

929 10961


,b_name,k_m,k_b,lift,lift_low
209590,Banana Bread Mix,6,38.0,0.334928,0.141802
209600,Chicken Sausage Apple,5,29.0,0.362838,0.140783
209626,Raisins Red Seedless Organic Jumbo Size,5,30.0,0.351134,0.136539
209705,Toasted Almond Rice Pilaf Mix,5,36.0,0.294193,0.115628
209717,Gluten Free Organic Peanut Butter & Chocolate ...,5,40.0,0.265491,0.104917


In [6]:
for t in [1, 2, 5, 10]:
    print(t, (ranked["lift_low"] > t).sum())

1 17428
2 692
5 133
10 42


In [7]:
cols = ["a_name", "b_name", "k_m", "lift", "lift_low"]
ranked[ranked["lift_low"].between(1, 1.5)].sample(10, random_state=42)[cols]

,a_name,b_name,k_m,lift,lift_low
10729,Natural Artesian Water,Bing Cherries,9,3.084173,1.099218
15580,Cucumber Kirby,Organic Chicken Noodle Soup,24,1.822157,1.022409
15232,Organic Lemon,Organic Baby Spinach,104,1.334746,1.026862
8142,Avocado Roll,Organic Granny Smith Apple,11,3.125854,1.160052
4539,Organic Lowfat 1% Milk,Dairy Free Coconut Milk Blueberry Yogurt Alter...,5,11.014986,1.287403
3431,Apples,Extra Fancy Unsalted Mixed Nuts,14,3.128652,1.356950
14726,Bag of Organic Bananas,Vanilla Bean Light Ice Cream,42,1.550896,1.034612
11638,Unsweetened Original Almond Breeze Almond Milk,Toasted Coconut Almondmilk Blend,5,9.259582,1.082128
5950,Unsalted Pure Irish Butter,Organic Whole Milk,17,2.673367,1.227711
8399,Roasted Turkey,Gluten-Free Chicken Nuggets,6,5.701277,1.152480


In [8]:
for t in [1, 2, 5]:
    print(t, ranked.loc[ranked["lift_low"] > t, "a"].nunique(), ranked["a"].nunique())

1 1399 2280
2 322 2280
5 88 2280


In [9]:
print(ranked["lift"].median())

1.3456790123456788


In [10]:
new_per_missing = totals["k_m"].sum() / totals["n_m"].sum()
new_per_bought = totals["k_b"].sum() / totals["n_b"].sum()
print(new_per_missing, new_per_bought, new_per_missing / new_per_bought)

4.1703659775932485 3.394297259100378 1.228638996308343


In [11]:
from src.pairs import adjust_for_exploration

ranked = adjust_for_exploration(ranked, totals)
print(ranked["lift_adj"].median())

for t in [1, 2]:
    keep = ranked["lift_low_adj"] > t
    print(t, keep.sum(), ranked.loc[keep, "a"].nunique())

exploration ratio: 1.229
1.0952598903249877
1 5928 973
2 423 231
